In [101]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

In [67]:
batting = pd.read_csv("batting.csv")

In [68]:
batting.head()

,Unnamed: 0,IDfg,Season,Name,Team,Age,G,AB,PA,H,...,Barrel%,maxEV,HardHit,HardHit%,Events,CStr%,CSW%,xBA,xSLG,xwOBA
0,0,1109,2002,Barry Bonds,SFG,37,143,403,612,149,...,NaN,NaN,NaN,NaN,0,0.127,0.191,NaN,NaN,NaN
1,1,1109,2004,Barry Bonds,SFG,39,147,373,617,135,...,NaN,NaN,NaN,NaN,0,0.124,0.164,NaN,NaN,NaN
2,15,13611,2018,Mookie Betts,BOS,25,136,520,614,180,...,0.131,110.6,217.0,0.5,434,0.220,0.270,NaN,NaN,NaN
3,2,1109,2003,Barry Bonds,SFG,38,130,390,550,133,...,NaN,NaN,NaN,NaN,0,0.135,0.223,NaN,NaN,NaN
4,78,10155,2013,Mike Trout,LAA,21,157,589,716,190,...,NaN,NaN,0.0,NaN,0,0.200,0.266,NaN,NaN,NaN


In [69]:
#Removing first column, not needed
batting.drop(columns=['Unnamed: 0'], inplace=True)

In [70]:
batting = batting.groupby("IDfg", group_keys = False).filter(lambda x: x.shape[0] > 1)

In [71]:
batting

,IDfg,Season,Name,Team,Age,G,AB,PA,H,1B,...,Barrel%,maxEV,HardHit,HardHit%,Events,CStr%,CSW%,xBA,xSLG,xwOBA
0,1109,2002,Barry Bonds,SFG,37,143,403,612,149,70,...,NaN,NaN,NaN,NaN,0,0.127,0.191,NaN,NaN,NaN
1,1109,2004,Barry Bonds,SFG,39,147,373,617,135,60,...,NaN,NaN,NaN,NaN,0,0.124,0.164,NaN,NaN,NaN
2,13611,2018,Mookie Betts,BOS,25,136,520,614,180,96,...,0.131,110.6,217.0,0.500,434,0.220,0.270,NaN,NaN,NaN
3,1109,2003,Barry Bonds,SFG,38,130,390,550,133,65,...,NaN,NaN,NaN,NaN,0,0.135,0.223,NaN,NaN,NaN
4,10155,2013,Mike Trout,LAA,21,157,589,716,190,115,...,NaN,NaN,0.0,NaN,0,0.200,0.266,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7084,1698,2010,Gerald Laird,DET,30,89,270,299,56,40,...,NaN,NaN,0.0,NaN,0,0.166,0.252,NaN,NaN,NaN
7086,9272,2018,Chris Davis,BAL,32,128,470,522,79,51,...,0.096,111.8,113.0,0.401,282,0.174,0.316,NaN,NaN,NaN
7087,319,2011,Adam Dunn,CHW,31,122,415,496,66,39,...,NaN,NaN,0.0,NaN,0,0.169,0.295,NaN,NaN,NaN
7088,620,2002,Neifi Perez,KCR,29,145,554,585,131,104,...,NaN,NaN,NaN,NaN,0,0.130,0.187,NaN,NaN,NaN


In [72]:
def next_season(player):
    player = player.sort_values("Season")
    player["Next_WAR"] = player["WAR"].shift(-1)
    return player

batting = batting.groupby("IDfg", group_keys = False).apply(next_season)

C:\Users\sporniaa\AppData\Local\Temp\ipykernel_7808\3454737146.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  batting = batting.groupby("IDfg", group_keys = False).apply(next_season)


In [73]:
batting[["Name", "Season", "WAR", "Next_WAR"]]

,Name,Season,WAR,Next_WAR
3934,Alfredo Amezaga,2006,1.1,2.0
2593,Alfredo Amezaga,2007,2.0,1.2
3759,Alfredo Amezaga,2008,1.2,NaN
1019,Garret Anderson,2002,3.7,5.1
427,Garret Anderson,2003,5.1,0.8
...,...,...,...,...
4837,Owen Miller,2022,0.6,NaN
6190,Andrew Vaughn,2021,-0.3,0.5
4887,Andrew Vaughn,2022,0.5,NaN
5038,Ha-seong Kim,2021,0.5,2.6


In [74]:
#Counting how many missing values are in each column

null_count = batting.isnull().sum()
null_count

IDfg           0
Season         0
Name           0
Team           0
Age            0
            ... 
CSW%           0
xBA         6737
xSLG        6737
xwOBA       6737
Next_WAR    1174
Length: 320, dtype: int64

In [75]:
#Selecting only columns where the null count is zero. Meaning there are no null values 

complete_cols = list(batting.columns[null_count == 0])

In [76]:
#We are taking our batting dataframe and selecting all the columns in the complete_cols list plus our Next_WAR column

batting = batting[complete_cols + ["Next_WAR"]].copy()

In [77]:
#Checking datatypes of columns. Need to make sure all columns are numeric, not strings.

batting.dtypes

IDfg          int64
Season        int64
Name         object
Team         object
Age           int64
             ...   
Hard%+        int64
Events        int64
CStr%       float64
CSW%        float64
Next_WAR    float64
Length: 132, dtype: object

In [78]:
#Finding the columns that are strings

batting.dtypes[batting.dtypes == "object"]

Name       object
Team       object
Dol        object
Age Rng    object
dtype: object

In [79]:
del batting["Dol"]

In [80]:
batting["Age Rng"]

3934    28 - 28
2593    29 - 29
3759    30 - 30
1019    30 - 30
427     31 - 31
         ...   
4837    25 - 25
6190    23 - 23
4887    24 - 24
5038    25 - 25
1892    26 - 26
Name: Age Rng, Length: 6737, dtype: object

In [81]:
#Doesnt provide value, we delete it

del batting["Age Rng"]

In [82]:
batting["Team"]

3934    FLA
2593    FLA
3759    FLA
1019    ANA
427     ANA
       ... 
4837    CLE
6190    CHW
4887    CHW
5038    SDP
1892    SDP
Name: Team, Length: 6737, dtype: object

In [83]:
#We are creating a new column called team code. From there we are assigning each team name a number, which converts it from a string to integer

batting["team_code"] = batting["Team"].astype("category").cat.codes

In [84]:
#Creating a copy of the dataframe

batting_full = batting.copy()

In [85]:
#Dropping any rows where the Next_WAR is None

batting = batting.dropna().copy()

In [86]:
#Setting alpha higher reduces overfitting and vicversa

rr = Ridge(alpha = 1)

#Splitting our data up to three parts and make predictions with those parts
split = TimeSeriesSplit(n_splits = 3)

#Will starrt by selection zero features and eveluate all features and find the best one, then again and find the best one and again until it has 20
sfs = SequentialFeatureSelector(rr, n_features_to_select = 20, direction = "forward", cv = split, n_jobs = 4)

In [87]:
#In order for our sequential selector to work we need to remove certain columns. We want to remove our predictor column and any text columns
#Also taking out season as we do not want it to overfit to a particular season
removed_columns = ["Next_WAR", "Name", "Team", "IDfg", "Season"]

#Take all of our columns in batting dataframe, then pick only the columns that arent in the removed columns list above
selected_columns = batting.columns[~batting.columns.isin(removed_columns)]

In [88]:
#Ensuring all our values are between zero and one
scaler = MinMaxScaler()
batting.loc[:, selected_columns] = scaler.fit_transform(batting[selected_columns])



C:\Users\sporniaa\AppData\Local\Temp\ipykernel_7808\2767615307.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.34615385 0.38461538 0.42307692 ... 0.19230769 0.15384615 0.23076923]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  batting.loc[:, selected_columns] = scaler.fit_transform(batting[selected_columns])
C:\Users\sporniaa\AppData\Local\Temp\ipykernel_7808\2767615307.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.73504274 0.74358974 0.95726496 ... 0.11965812 0.69230769 0.60683761]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  batting.loc[:, selected_columns] = scaler.fit_transform(batting[selected_columns])
C:\Users\sporniaa\AppData\Local\Temp\ipykernel_7808\2767615307.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated 

In [89]:
#Scaling completed
batting.head()

,IDfg,Season,Name,Team,Age,G,AB,PA,H,1B,...,Cent%+,Oppo%+,Soft%+,Med%+,Hard%+,Events,CStr%,CSW%,Next_WAR,team_code
3934,1,2006,Alfredo Amezaga,FLA,0.346154,0.735043,0.312950,0.307958,0.245690,0.278302,...,0.539326,0.503759,0.662921,0.652174,0.210884,0.0,0.582979,0.524229,2.0,0.352941
2593,1,2007,Alfredo Amezaga,FLA,0.384615,0.743590,0.431655,0.429066,0.323276,0.316038,...,0.471910,0.496241,0.471910,0.710145,0.292517,0.0,0.527660,0.396476,1.2,0.352941
1019,2,2002,Garret Anderson,ANA,0.423077,0.957265,0.859712,0.826990,0.711207,0.443396,...,0.359551,0.255639,0.224719,0.478261,0.659864,0.0,0.365957,0.418502,5.1,0.029412
427,2,2003,Garret Anderson,ANA,0.461538,0.965812,0.859712,0.818339,0.737069,0.500000,...,0.471910,0.255639,0.365169,0.507246,0.523810,0.0,0.480851,0.506608,0.8,0.029412
4349,2,2004,Garret Anderson,ANA,0.500000,0.564103,0.507194,0.475779,0.443966,0.400943,...,0.494382,0.218045,0.297753,0.608696,0.448980,0.0,0.531915,0.585903,-0.2,0.029412


In [90]:
#Using our sequential function to have it pick our 20 predictors with the greatest accuracy
sfs.fit(batting[selected_columns], batting["Next_WAR"])

,estimator,Ridge(alpha=1)
,n_features_to_select,20
,tol,None
,direction,'forward'
,scoring,None
,cv,TimeSeriesSpl...est_size=None)
,n_jobs,4
,alpha,1
,fit_intercept,True
,copy_X,True
,max_iter,None


In [91]:
#Seeing the list of the 20 predictors that our sequential selector picked, showing as True or False
sfs.get_support()

array([ True, False, False, False, False, False, False, False, False,
       False, False, False,  True,  True, False, False, False, False,
        True, False, False, False, False, False, False, False, False,
       False, False,  True, False, False, False, False, False, False,
       False, False,  True, False, False, False, False, False, False,
        True, False, False, False, False, False, False, False, False,
        True,  True, False, False, False, False, False, False, False,
        True, False, False, False, False, False, False,  True, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False,  True, False, False, False,
        True, False, False, False, False, False, False,  True, False,
       False,  True, False, False, False, False, False, False, False,
       False,  True, False, False,  True, False, False, False,  True,
       False, False,  True, False,  True, False, False, False, False])

In [94]:
#Sving our predictors in a variable
#Now we pass our selected columns list into this get_support function so we are able to see exactly what the column names are
predictors = list(selected_columns[sfs.get_support()])

In [98]:
#We only want to use past data to preict future data. We do not want to use 2024 data to help predict 2023 data
def backtest(data, model, predictors, start = 5, step = 1):
    all_predictions = []

    years = sorted(data["Season"].unique())
    
    #Each time in this loop we are saying to use a historical season to predict a new one
    #In this case we will start with 2007, giving us a few years before we start predicting. 
    #2002 - 2007 as training data and predict 2008, then 2002 - 2008 as training data and predict 2009 and so on
    for i in range(start, len(years), step):
        current_year = years[i]

        train = data[data["Season"] < current_year]
        test = data[data["Season"] == current_year]

        model.fit(train[predictors], train["Next_WAR"])

        preds = model.predict(test[predictors])
        #preds returns the predictions in a numpy array, we want to convert to pandas series
        preds = pd.Series(preds, index = test.index)
        #Creating 2 seperate columns to compare
        combined = pd.concat([test["Next_WAR"], preds], axis = 1)
        #Assignging column names
        combined.columns = ["actual", "prediction"]

        #Appending all yearly predictions to the all_predictins empty list
        all_predictions.append(combined)
    #This is combining all the dataframes vertically, ontop of each other, to create one long dataframe
    return pd.concat(all_predictions)

In [99]:
predictions = backtest(batting, rr, predictors)

In [100]:
predictions

,actual,prediction
2593,1.2,1.514187
3367,1.4,0.804184
4554,-0.1,0.587281
4647,0.6,0.890092
1741,4.8,2.307446
...,...,...
2051,1.2,2.697911
4626,1.0,1.926963
6861,0.6,1.545744
6190,0.5,1.646229


In [103]:
#This will subtract the prediction from the atual value, and then square the difference, and then find the average squared difference across all rows
mean_squared_error(predictions["actual"], predictions["prediction"])

2.7671807143292715

In [104]:
#Good to have the mean squared error be lower then the standard deviation
batting["Next_WAR"].describe()

count    5563.000000
mean        1.787758
std         1.989465
min        -3.400000
25%         0.300000
50%         1.500000
75%         2.900000
max        11.900000
Name: Next_WAR, dtype: float64

In [105]:
2.7671807143292715 ** .5

1.6634845097954087

In [106]:
#Giving the algorithm information on how the player did on a previous season can help it make better preidctions
def player_history(df):
    #First we sort by season so its in order
    df = df.sort_values("Season")
    
    #This here tells us which season it is for each player, is it there first season or 4th season
    df["player_season"] = range(0, df.shape[0])
    df["war_corr"] = list(df[["player_season", "WAR"]].expanding().corr().loc[(slice(None), "player_season"), "WAR"])
    df["war_corr"].fillna(1, inplace = True)

    #Takes WAR for current season and divdes it against previous season
    df["war_diff"] = df["WAR"] / df["WAR"].shift(1)
    df["war_diff"].fillna(1, inplace = True)

    #Find any values in war_diff that are infinite and replace them as 1
    df["war_diff"][df["war_diff"] == np.inf] = 1

    return df

In [107]:

#Splitting our dataframe up into groups by player and for each player its calling the player history function and passing in data for that player
batting = batting.groupby("IDfg", group_keys = False).apply(player_history)

C:\Users\sporniaa\AppData\Local\Temp\ipykernel_7808\2135600871.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["war_corr"].fillna(1, inplace = True)
C:\Users\sporniaa\AppData\Local\Temp\ipykernel_7808\2135600871.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exampl

In [111]:
def group_averages(df):
    #Did our player perform better or worse against the average WAR
    return df["WAR"] / df["WAR"].mean()

In [112]:
batting["war_season"] = batting.groupby("Season", group_keys = False).apply(group_averages)

C:\Users\sporniaa\AppData\Local\Temp\ipykernel_7808\1991322583.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  batting["war_season"] = batting.groupby("Season", group_keys = False).apply(group_averages)


In [113]:
new_predictors = predictors + ["player_season", "war_corr", "war_season", "war_diff"]

In [114]:
predictions = backtest(batting, rr, new_predictors)

In [115]:
mean_squared_error(predictions["actual"], predictions["prediction"])

2.670955561949691

In [117]:
#Anyhing with a small coef the model isnt really taking into account but anything large the model is taking it largely into account
pd.Series(rr.coef_, index = new_predictors).sort_values()

Age             -2.725100
WAR             -1.788656
BABIP           -1.534478
Soft%+          -1.256893
SLG+            -1.227397
SwStr%          -1.047389
BU              -0.975975
PH              -0.738206
SO              -0.707326
Z-Contact%      -0.695398
war_diff        -0.586509
wGDP            -0.477138
Pull%+          -0.231475
LD+%            -0.223573
CB%             -0.214705
war_corr        -0.122821
player_season    0.000056
IFH%             0.380017
Oppo%            0.660879
Spd              0.717269
SB               1.022885
IBB              1.757543
Hard%+           2.256642
war_season       3.436497
dtype: float64